Data setup

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('/content/creditcard.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53571 entries, 0 to 53570
Data columns (total 31 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Time    53571 non-null  int64  
 1   V1      53571 non-null  float64
 2   V2      53571 non-null  float64
 3   V3      53571 non-null  float64
 4   V4      53571 non-null  float64
 5   V5      53571 non-null  float64
 6   V6      53571 non-null  float64
 7   V7      53571 non-null  float64
 8   V8      53571 non-null  float64
 9   V9      53571 non-null  float64
 10  V10     53571 non-null  float64
 11  V11     53571 non-null  float64
 12  V12     53571 non-null  float64
 13  V13     53571 non-null  float64
 14  V14     53571 non-null  float64
 15  V15     53571 non-null  float64
 16  V16     53571 non-null  float64
 17  V17     53571 non-null  float64
 18  V18     53571 non-null  float64
 19  V19     53571 non-null  float64
 20  V20     53571 non-null  float64
 21  V21     53571 non-null  float64
 22

In [4]:
df.isnull()
df.isnull().sum()

,0
Time,0
V1,0
V2,0
V3,0
V4,0
V5,0
V6,0
V7,0
V8,0
V9,0


In [5]:
df = df.dropna()

Assgining values

In [6]:
x = df.drop('Class', axis = 1)
y = df['Class']

In [7]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 0)

Model

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [9]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(),
    'Decision Tree': DecisionTreeClassifier(),
    'Support Vector Machine': SVC()
}

In [10]:
from sklearn.metrics import accuracy_score

for name,model in models.items():
  # train
  model.fit(x_train, y_train)
  # prediction
  y_pred = model.predict(x_test)
  # evaluation
  accuracy = accuracy_score(y_test, y_pred)
  print(f'{name} accuracy: {accuracy}')

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression accuracy: 0.9976666044427851
Random Forest accuracy: 0.9994399850662684
Decision Tree accuracy: 0.9985999626656711
Support Vector Machine accuracy: 0.997013253686765


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 53570 entries, 0 to 53569
Data columns (total 31 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Time    53570 non-null  int64  
 1   V1      53570 non-null  float64
 2   V2      53570 non-null  float64
 3   V3      53570 non-null  float64
 4   V4      53570 non-null  float64
 5   V5      53570 non-null  float64
 6   V6      53570 non-null  float64
 7   V7      53570 non-null  float64
 8   V8      53570 non-null  float64
 9   V9      53570 non-null  float64
 10  V10     53570 non-null  float64
 11  V11     53570 non-null  float64
 12  V12     53570 non-null  float64
 13  V13     53570 non-null  float64
 14  V14     53570 non-null  float64
 15  V15     53570 non-null  float64
 16  V16     53570 non-null  float64
 17  V17     53570 non-null  float64
 18  V18     53570 non-null  float64
 19  V19     53570 non-null  float64
 20  V20     53570 non-null  float64
 21  V21     53570 non-null  float64
 22  V22

LSTM branch

In [12]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [13]:
x_train_lstm = x_train.values.reshape((x_train.shape[0], x_train.shape[1], 1))
x_test_lstm = x_test.values.reshape((x_test.shape[0], x_test.shape[1], 1))

In [14]:
model = Sequential()
model.add(LSTM(units = 64, return_sequences = True, input_shape = (x_train.shape[1], 1), activation='tanh'))
model.add(Dropout(0.2))
model.add(LSTM(units = 32, activation='tanh'))
model.add(Dropout(0.2))
model.add(Dense(units = 1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [15]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [16]:
# class weights
fraud_count  = (y_train == 1).sum()
normal_count = (y_train == 0).sum()
class_weight = {0: 1.0, 1: normal_count / fraud_count}

In [17]:
history = model.fit(x_train_lstm, y_train,
                    epochs=10,
                    batch_size=256,
                    validation_split=0.1,
                    class_weight=class_weight,
                    verbose=1)

Epoch 1/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.9802 - loss: 0.4846 - val_accuracy: 0.9942 - val_loss: 0.0582
Epoch 2/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9919 - loss: 0.3272 - val_accuracy: 0.9893 - val_loss: 0.0930
Epoch 3/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9962 - loss: 0.2670 - val_accuracy: 0.9949 - val_loss: 0.0654
Epoch 4/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9886 - loss: 0.2584 - val_accuracy: 0.9958 - val_loss: 0.0510
Epoch 5/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9954 - loss: 0.2402 - val_accuracy: 0.9900 - val_loss: 0.0939
Epoch 6/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9916 - loss: 0.2466 - val_accuracy: 0.9937 - val_loss: 0.0699
Epoch 7/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9830 - loss: 0.2287 - val_accuracy: 0.9965 - val_loss: 0.0625
Epoch 8/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9871 - loss: 0.2136 - val_accuracy:

Evaluation

In [18]:
from sklearn.metrics import classification_report

y_pred_lstm = (model.predict(x_test_lstm) > 0.5).astype(int)
accuracy    = (y_pred_lstm.flatten() == y_test).mean()
print(f'LSTM accuracy: {accuracy}')
print(classification_report(y_test, y_pred_lstm, target_names=['Normal', 'Fraud']))

335/335 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
LSTM accuracy: 0.9887063655030801
              precision    recall  f1-score   support

      Normal       1.00      0.99      0.99     10682
       Fraud       0.20      0.94      0.33        32

    accuracy                           0.99     10714
   macro avg       0.60      0.96      0.66     10714
weighted avg       1.00      0.99      0.99     10714



In [22]:
from google.colab import drive
drive.mount('/content/drive')

model.save('/content/drive/MyDrive/lstm_fraud_model.keras')

Mounted at /content/drive


Live demo creation

In [33]:
%%writefile app.py
from flask import Flask, request, jsonify
from flask_cors import CORS
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
import pandas as pd

app = Flask(__name__)
CORS(app)

model = None
scaler = None

def load_model_and_scaler():
    global model, scaler
    model = tf.keras.models.load_model('/content/drive/MyDrive/lstm_fraud_model.keras')
    print("✓ Model loaded")
    df = pd.read_csv('/content/creditcard.csv').dropna()
    scaler = StandardScaler()
    scaler.fit(df.drop('Class', axis=1).drop('Time', axis=1))
    print("✓ Scaler fitted")

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'model_loaded': model is not None})

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        features = np.array(data['features'], dtype=np.float32).reshape(1, -1)
        features_scaled = scaler.transform(features)
        features_lstm = features_scaled.reshape(1, 1, features_scaled.shape[1])
        prob = float(model.predict(features_lstm, verbose=0)[0][0])
        fraud = prob > 0.5
        return jsonify({
            'fraud': fraud,
            'confidence': round(prob if fraud else 1 - prob, 4),
            'label': 'FRAUD' if fraud else 'CLEAR'
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 400

Overwriting app.py


In [36]:
from google.colab.output import eval_js
# load_model_and_scaler()
print(eval_js("google.colab.kernel.proxyPort(5000)"))
# app.run(port=5000)

https://5000-gpu-t4-s-kkb-usw4b2-11mlzmn15hoen-b.us-west4-2.prod.colab.dev


In [26]:
# install dependencies
!pip install flask flask-cors pyngrok -q

In [27]:
!python app.py

2026-07-16 02:06:00.617126: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1784167560.626260    3576 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13549 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
✓ Model loaded
✓ Scaler fitted
t=2026-07-16T02:06:12+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:  authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a sessio

In [19]:
get_ipython().system('git clone https://github.com/neural-shubh/credit-card-fraud.git')
get_ipython().system('cd credit-card-fraud')

Cloning into 'credit-card-fraud'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 15 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 9.95 KiB | 9.95 MiB/s, done.
Resolving deltas: 100% (2/2), done.
